In [ ]:
pip install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.5/114.5 kB 10.4 MB/s eta 0:00:00


In [2]:
import ijson
import pandas as pd
import tensorflow as tf
import numpy as np

class DotA_Data:

    def process_teams_and_one_hot_encode(self, features, hero_id_columns, is_radiant_columns, extra_column_indices, num_heroes=140):
        # Initialize zero-filled vectors for Radiant and Dire teams
        radiant_vector = np.zeros(num_heroes)
        dire_vector = np.zeros(num_heroes)

        for i in range(len(hero_id_columns)):
            hero_id = int(features[i])  # Ensure hero ID is an integer
            is_radiant = int(features[len(hero_id_columns) + i])

            # Set the corresponding position in the vector based on team
            if is_radiant == 1:
                radiant_vector[hero_id] = 1
            else:
                dire_vector[hero_id] = 1

        # Concatenate the two vectors to form a 280-length vector
        match_vector = np.concatenate([radiant_vector, dire_vector])

        # Collect extra features, ensuring consistency in length
        extra_features = [float(features[extra_col]) if extra_col < len(features) else 0.0 for extra_col in extra_column_indices]

        # Convert extra_features to numpy array and concatenate with match_vector
        extra_features = np.array(extra_features, dtype=np.float32)
        match_vector = np.concatenate([match_vector, extra_features])

        return match_vector



    def create_one_hot_encoded_chunk(self, chunk_df, hero_id_columns, is_radiant_columns, extra_columns, target_column='radiant_win', num_heroes=140):
        extra_column_indices = [chunk_df.columns.get_loc(col) for col in extra_columns]

        feature_vectors = []
        targets = []

        for _, row in chunk_df.iterrows():
            features = row[hero_id_columns + is_radiant_columns + extra_columns].values
            target = row[target_column]

            # Ensure consistent length by using the modified function
            feature_vector = self.process_teams_and_one_hot_encode(features, hero_id_columns, is_radiant_columns, extra_column_indices, num_heroes)
            feature_vectors.append(feature_vector)
            targets.append(target)

        # Convert lists to numpy arrays with consistent shape for TensorFlow
        feature_vectors = np.array(feature_vectors, dtype=np.float32)
        targets = np.array(targets, dtype=np.int32)

        return tf.data.Dataset.from_tensor_slices((feature_vectors, targets))



    def prepare_dataset(self, dataset_path, hero_id_columns, is_radiant_columns, extra_columns, target_column='radiant_win', chunksize=500):
        final_dataset = None
        print("Processing data in chunks to build TensorFlow dataset")

        with open(dataset_path, 'r') as f:
            # Use ijson to stream the JSON data, this avoids loading the entire file into memory at once
            objects = ijson.items(f, 'item')  # Adjust 'item' based on your JSON structure

            chunk = []
            for i, obj in enumerate(objects):
                chunk.append(obj)

                if len(chunk) == chunksize:
                    # Once chunk size is met, process and clear the chunk
                    chunk_df = pd.json_normalize(chunk)
                    chunk_dataset = self.create_one_hot_encoded_chunk(chunk_df, hero_id_columns, is_radiant_columns, extra_columns, target_column)

                    # Concatenate the chunk dataset with the final dataset
                    if final_dataset is None:
                        final_dataset = chunk_dataset
                    else:
                        final_dataset = final_dataset.concatenate(chunk_dataset)

                    chunk = []  # Clear the chunk for the next batch

            # Process any remaining data after the loop
            if chunk:
                chunk_df = pd.json_normalize(chunk)
                chunk_dataset = self.create_one_hot_encoded_chunk(chunk_df, hero_id_columns, is_radiant_columns, extra_columns, target_column)

                if final_dataset is None:
                    final_dataset = chunk_dataset
                else:
                    final_dataset = final_dataset.concatenate(chunk_dataset)

        return final_dataset

In [3]:
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

import pandas as pd
import numpy as np


class DotA_NN:

    input_variables = ['players_0_hero_id', 'players_1_hero_id', 'players_2_hero_id', 'players_3_hero_id', 'players_4_hero_id', 'players_5_hero_id', 'players_6_hero_id', 'players_7_hero_id', 'players_8_hero_id', 'players_9_hero_id', 'players_0_isRadiant', 'players_1_isRadiant', 'players_2_isRadiant', 'players_3_isRadiant', 'players_4_isRadiant', 'players_5_isRadiant', 'players_6_isRadiant', 'players_7_isRadiant', 'players_8_isRadiant', 'players_9_isRadiant']

    def __init__(self):
        # Set up the strategy for multi-GPU support
        self.strategy = tf.distribute.MirroredStrategy()
        print('Number of devices: {}'.format(self.strategy.num_replicas_in_sync))
        print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

    def split_data(self, dataset, train_ratio=0.6, seed=42):
        val_ratio = test_ratio = (1 - train_ratio) / 2
        # Ensure ratios add up to 1.0
        assert train_ratio + val_ratio + test_ratio == 1.0, "Ratios must sum to 1.0"

        # Calculate dataset size
        dataset_size = dataset.cardinality().numpy()

        # Compute sizes based on ratios
        train_size = int(train_ratio * dataset_size)
        val_size = int(val_ratio * dataset_size)

        #Split dataset using sizes
        train_dataset = dataset.take(train_size)
        val_dataset = dataset.skip(train_size).take(val_size)
        test_dataset = dataset.skip(train_size + val_size)

        return train_dataset, val_dataset, test_dataset

    def build_model(self, input_size, learning_rate=0.001):
        hidden_layer_fn = 'relu'
        with self.strategy.scope():
            model = Sequential([
                Flatten(),  # Dynamically flatten the input (no need to specify input_shape)
                Dense(256, activation=hidden_layer_fn),
                Dense(256, activation=hidden_layer_fn),
                Dense(256, activation=hidden_layer_fn),
                Dense(256, activation=hidden_layer_fn),
                Dense(256, activation=hidden_layer_fn),
                Dense(128, activation=hidden_layer_fn),
                Dense(64, activation=hidden_layer_fn),
                Dense(32, activation=hidden_layer_fn),
                Dense(1, activation='sigmoid')
            ])

            model.compile(optimizer=Adam(learning_rate=learning_rate), loss='binary_crossentropy', metrics=['accuracy'])

        return model

    def plot_accuracy_vs_epochs(self, model_history):
        plt.plot(model_history.history['accuracy'])
        plt.title('model accuracy')
        plt.ylabel('accuracy')
        plt.xlabel('epoch')
        plt.savefig('NN accuracy_vs_epochs.png')

    def inspect_first_row(self, dataset, output_file='first_row.txt'):
        # Take the first element from the dataset
        first_row = next(iter(dataset.take(1)))

        features, label = first_row  # Unpack the features and the label

        # Convert the Tensors to NumPy arrays for easier reading
        features_np = features.numpy()
        label_np = label.numpy()

        # Write the values to a text file
        with open(output_file, 'w') as f:
            f.write("Features (X):\n")
            np.savetxt(f, features_np, fmt='%s')
            f.write("\nLabel (y):\n")
            f.write(f"{label_np}\n")

        print(f"First row saved to {output_file}")

    def train(self, dataset, train_size, epochs, batch_size, learning_rate):
        self.inspect_first_row(dataset, output_file='first_row.txt')

        print("Splitting data")
        train_dataset, val_dataset, test_dataset = self.split_data(dataset, train_size)

        self.inspect_first_row(train_dataset, output_file='first_row_after_split.txt')

        # Check if the dataset is empty
        print(f"Train dataset size: {train_dataset.cardinality().numpy()}")
        if train_dataset.cardinality().numpy() == 0:
            raise ValueError("Train dataset is empty!")

        print(f"Val dataset size: {val_dataset.cardinality().numpy()}")
        if val_dataset.cardinality().numpy() == 0:
            raise ValueError("Train dataset is empty!")

        print(f"Test dataset size: {test_dataset.cardinality().numpy()}")
        if test_dataset.cardinality().numpy() == 0:
            raise ValueError("Train dataset is empty!")

        # Prepare datasets for batching and prefetching
        train_dataset = train_dataset.batch(batch_size, drop_remainder=True).prefetch(tf.data.experimental.AUTOTUNE)
        val_dataset = val_dataset.batch(batch_size, drop_remainder=True).prefetch(tf.data.experimental.AUTOTUNE)
        test_dataset = test_dataset.batch(batch_size, drop_remainder=True).prefetch(tf.data.experimental.AUTOTUNE)

        model = self.build_model(input_size=next(iter(train_dataset))[0].shape[1], learning_rate=learning_rate)

        print("Begin model training")
        model_history = model.fit(train_dataset, epochs=epochs, validation_data=val_dataset, verbose=0)

        print("Model training complete")
        # Save the model
        model.save('dota_nn_model.keras')
        print("Model saved as 'dota_nn_model.keras'")

        print("Hyperparameter settings: \n", "train_size: ", train_size, "\n", "epochs: ", epochs, "\n", "batch_size: ", batch_size, "\n", "learning rate: ", learning_rate)
        test_loss, test_accuracy = model.evaluate(test_dataset, verbose=0)
        print('\nTest accuracy:', test_accuracy)
        print('\nLoss:', test_loss)

        precision_metric = tf.keras.metrics.Precision()
        for features, labels in test_dataset:
            predictions = model.predict(features)
            precision_metric.update_state(labels, predictions)
        precision = precision_metric.result().numpy()
        print('\nPrecision:', precision)

        auc_metric = tf.keras.metrics.AUC()
        for features, labels in test_dataset:
            predictions = model.predict(features)
            auc_metric.update_state(labels, predictions)
        roc_auc = auc_metric.result().numpy()
        print('\nROC AUC:', roc_auc)

        self.plot_accuracy_vs_epochs(model_history)

        return test_accuracy, roc_auc, precision, test_loss

In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Bidirectional, Dropout
import numpy as np
import matplotlib.pyplot as plt


class DotA_Stacked_BiLSTM:

    def __init__(self):
        # Set up the strategy for multi-GPU support
        self.strategy = tf.distribute.MirroredStrategy()
        print('Number of devices: {}'.format(self.strategy.num_replicas_in_sync))
        print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

    def build_model(self, input_shape):
        with self.strategy.scope():
            model = Sequential()

            # Use LSTM units that match the input feature size dynamically
            input_feature_size = input_shape[-1]  # Get the feature dimension from the input shape

            # Add first Bidirectional LSTM layer with units matching input feature size
            model.add(Bidirectional(LSTM(input_feature_size, return_sequences=True), input_shape=(None, input_feature_size)))
            model.add(Dropout(0.01))

            # Add second Bidirectional LSTM layer
            model.add(Bidirectional(LSTM(input_feature_size, return_sequences=False)))  # No need to keep sequence data here
            model.add(Dropout(0.01))

            # Final Dense layer for binary classification
            model.add(Dense(1, activation='sigmoid'))  # Single sigmoid output for binary classification

            # Compile the model
            model.compile(optimizer='adam',
                        loss='binary_crossentropy',
                        metrics=['accuracy'])

        return model

    def split_data(self, dataset, train_ratio=0.6, seed=42):
        val_ratio = test_ratio = (1 - train_ratio) / 2
        # Ensure ratios add up to 1.0
        assert train_ratio + val_ratio + test_ratio == 1.0, "Ratios must sum to 1.0"

        # Calculate dataset size
        dataset_size = dataset.cardinality().numpy()

        # Compute sizes based on ratios
        train_size = int(train_ratio * dataset_size)
        val_size = int(val_ratio * dataset_size)

        # Create splits
        train_dataset = dataset.take(train_size)
        val_dataset = dataset.skip(train_size).take(val_size)
        test_dataset = dataset.skip(train_size + val_size)

        return train_dataset, val_dataset, test_dataset

    def inspect_first_row(self, dataset, output_file='first_row.txt'):
        # Take the first element from the dataset
        first_row = next(iter(dataset.take(1)))

        features, label = first_row  # Unpack the features and the label

        # Convert the Tensors to NumPy arrays for easier reading
        features_np = features.numpy()
        label_np = label.numpy()

        # Write the values to a text file
        with open(output_file, 'w') as f:
            f.write("Features (X):\n")
            np.savetxt(f, features_np, fmt='%s')
            f.write("\nLabel (y):\n")
            f.write(f"{label_np}\n")

        print(f"First row saved to {output_file}")

    def plot_accuracy_vs_epochs(self, model_history):
        plt.plot(model_history.history['accuracy'])
        plt.title('model accuracy')
        plt.ylabel('accuracy')
        plt.xlabel('epoch')
        plt.savefig('LSTM accuracy_vs_epochs.png')

    def train(self, dataset, train_size, epochs, batch_size):
        print("Splitting data")
        train_dataset, val_dataset, test_dataset = self.split_data(dataset, train_size)

        self.inspect_first_row(train_dataset, output_file='first_row_after_split_lstm.txt')

        # Check if the dataset is empty
        print(f"Train dataset size: {train_dataset.cardinality().numpy()}")
        if train_dataset.cardinality().numpy() == 0:
            raise ValueError("Train dataset is empty!")

        print(f"Val dataset size: {val_dataset.cardinality().numpy()}")
        if val_dataset.cardinality().numpy() == 0:
            raise ValueError("Train dataset is empty!")

        print(f"Test dataset size: {test_dataset.cardinality().numpy()}")
        if test_dataset.cardinality().numpy() == 0:
            raise ValueError("Train dataset is empty!")

        # Prepare datasets for batching and prefetching
        train_dataset = train_dataset.batch(batch_size, drop_remainder=True).prefetch(tf.data.experimental.AUTOTUNE)
        val_dataset = val_dataset.batch(batch_size, drop_remainder=True).prefetch(tf.data.experimental.AUTOTUNE)
        test_dataset = test_dataset.batch(batch_size, drop_remainder=True).prefetch(tf.data.experimental.AUTOTUNE)

        input_shape = next(iter(train_dataset))[0].shape[1:]  # Get input shape from the first batch
        print(f"Dynamically inferred input shape: {input_shape}")

        # Ensure the input data has the sequence dimension (adding a dummy sequence dimension if needed)
        train_dataset = train_dataset.map(lambda x, y: (tf.expand_dims(x, 1), y))  # Adding sequence length dimension (1)
        val_dataset = val_dataset.map(lambda x, y: (tf.expand_dims(x, 1), y))  # Same for validation data
        test_dataset = test_dataset.map(lambda x, y: (tf.expand_dims(x, 1), y))  # Same for test data

        model = self.build_model(input_shape)

        print("Begin Training")
        model_history = model.fit(train_dataset, validation_data=val_dataset, epochs=epochs, verbose=0)

        print("Stacked BiLSTM Model training complete")
        # Save the model
        model.save('dota_lstm_model.keras')
        print("Model saved as 'dota_lstm_model.keras'")

        print("Hyperparameter settings: \n", "train_size: ", train_size, "\n", "epochs: ", epochs, "\n", "batch_size: ", batch_size)
        test_loss, test_accuracy = model.evaluate(test_dataset, verbose=0)
        print('\nTest accuracy:', test_accuracy)
        print('\nLoss:', test_loss)

        precision_metric = tf.keras.metrics.Precision()
        for features, labels in test_dataset:
            predictions = model.predict(features, verbose=0)
            precision_metric.update_state(labels, predictions)
        precision = precision_metric.result().numpy()
        print('\nPrecision:', precision)

        auc_metric = tf.keras.metrics.AUC()
        for features, labels in test_dataset:
            predictions = model.predict(features, verbose=0)
            auc_metric.update_state(labels, predictions)
        roc_auc = auc_metric.result().numpy()
        print('\nROC AUC:', roc_auc)

        self.plot_accuracy_vs_epochs(model_history)

        return test_accuracy, roc_auc, precision, test_loss

In [ ]:
# from DotA_Data import DotA_Data
# from DotA_NN import DotA_NN
# from DotA_Stacked_BiLSTM import DotA_Stacked_BiLSTM
# from DotA_Postgre_Data import DotA_Postgre_Data
from pprint import pprint
import json
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

#json_file_path = '/home-mscluster/rlazarus/flattened_postgre_dota_data.json'
#json_file_path = '/home-mscluster/rlazarus/flattened_dota_data.json'
#json_file_path = 'C:/Users/Reece Lazarus/DOTA2_Match_Prediction/flattened_postgre_dota_data.json'
#json_file_path = 'C:/Users/Reece Lazarus/DOTA2_Match_Prediction/flattened_dota_data.json'

#json_file_path = '/content/drive/MyDrive/research_data/flattened_dota_data.json'
json_file_path = '/content/drive/MyDrive/research_data/flattened_postgre_dota_data.json'

chunksize = 100

hero_id_columns = [
    'players_0_hero_id', 'players_1_hero_id', 'players_2_hero_id', 'players_3_hero_id', 'players_4_hero_id',
    'players_5_hero_id', 'players_6_hero_id', 'players_7_hero_id', 'players_8_hero_id', 'players_9_hero_id'
    ]

is_radiant_columns = [
        'players_0_isRadiant', 'players_1_isRadiant', 'players_2_isRadiant', 'players_3_isRadiant', 'players_4_isRadiant',
        'players_5_isRadiant', 'players_6_isRadiant', 'players_7_isRadiant', 'players_8_isRadiant', 'players_9_isRadiant'
    ]

extra_columns = [
    # Gold and XP per minute
    'players_0_gold_per_min', 'players_1_gold_per_min', 'players_2_gold_per_min', 'players_3_gold_per_min',
    'players_4_gold_per_min', 'players_5_gold_per_min', 'players_6_gold_per_min', 'players_7_gold_per_min',
    'players_8_gold_per_min', 'players_9_gold_per_min',
    'players_0_xp_per_min', 'players_1_xp_per_min', 'players_2_xp_per_min', 'players_3_xp_per_min',
    'players_4_xp_per_min', 'players_5_xp_per_min', 'players_6_xp_per_min', 'players_7_xp_per_min',
    'players_8_xp_per_min', 'players_9_xp_per_min',

    # Player slot
    'players_0_player_slot', 'players_1_player_slot', 'players_2_player_slot', 'players_3_player_slot',
    'players_4_player_slot', 'players_5_player_slot', 'players_6_player_slot', 'players_7_player_slot',
    'players_8_player_slot', 'players_9_player_slot',

    # Kills, deaths, assists
    'players_0_kills', 'players_1_kills', 'players_2_kills', 'players_3_kills', 'players_4_kills',
    'players_5_kills', 'players_6_kills', 'players_7_kills', 'players_8_kills', 'players_9_kills',
    'players_0_deaths', 'players_1_deaths', 'players_2_deaths', 'players_3_deaths', 'players_4_deaths',
    'players_5_deaths', 'players_6_deaths', 'players_7_deaths', 'players_8_deaths', 'players_9_deaths',
    'players_0_assists', 'players_1_assists', 'players_2_assists', 'players_3_assists', 'players_4_assists',
    'players_5_assists', 'players_6_assists', 'players_7_assists', 'players_8_assists', 'players_9_assists',

    # Last hits and denies
    'players_0_last_hits', 'players_1_last_hits', 'players_2_last_hits', 'players_3_last_hits', 'players_4_last_hits',
    'players_5_last_hits', 'players_6_last_hits', 'players_7_last_hits', 'players_8_last_hits', 'players_9_last_hits',
    'players_0_denies', 'players_1_denies', 'players_2_denies', 'players_3_denies', 'players_4_denies',
    'players_5_denies', 'players_6_denies', 'players_7_denies', 'players_8_denies', 'players_9_denies',

    # Level and net worth
    'players_0_level', 'players_1_level', 'players_2_level', 'players_3_level', 'players_4_level',
    'players_5_level', 'players_6_level', 'players_7_level', 'players_8_level', 'players_9_level',
    'players_0_net_worth', 'players_1_net_worth', 'players_2_net_worth', 'players_3_net_worth', 'players_4_net_worth',
    'players_5_net_worth', 'players_6_net_worth', 'players_7_net_worth', 'players_8_net_worth', 'players_9_net_worth',

    # Items: Aghanim's Scepter, Shard, Moonshard
    'players_0_aghanims_scepter', 'players_1_aghanims_scepter', 'players_2_aghanims_scepter', 'players_3_aghanims_scepter',
    'players_4_aghanims_scepter', 'players_5_aghanims_scepter', 'players_6_aghanims_scepter', 'players_7_aghanims_scepter',
    'players_8_aghanims_scepter', 'players_9_aghanims_scepter',
    'players_0_aghanims_shard', 'players_1_aghanims_shard', 'players_2_aghanims_shard', 'players_3_aghanims_shard',
    'players_4_aghanims_shard', 'players_5_aghanims_shard', 'players_6_aghanims_shard', 'players_7_aghanims_shard',
    'players_8_aghanims_shard', 'players_9_aghanims_shard',
    'players_0_moonshard', 'players_1_moonshard', 'players_2_moonshard', 'players_3_moonshard', 'players_4_moonshard',
    'players_5_moonshard', 'players_6_moonshard', 'players_7_moonshard', 'players_8_moonshard', 'players_9_moonshard',

    # Hero, tower, and healing damage
    'players_0_hero_damage', 'players_1_hero_damage', 'players_2_hero_damage', 'players_3_hero_damage', 'players_4_hero_damage',
    'players_5_hero_damage', 'players_6_hero_damage', 'players_7_hero_damage', 'players_8_hero_damage', 'players_9_hero_damage',
    'players_0_tower_damage', 'players_1_tower_damage', 'players_2_tower_damage', 'players_3_tower_damage', 'players_4_tower_damage',
    'players_5_tower_damage', 'players_6_tower_damage', 'players_7_tower_damage', 'players_8_tower_damage', 'players_9_tower_damage',
    'players_0_hero_healing', 'players_1_hero_healing', 'players_2_hero_healing', 'players_3_hero_healing', 'players_4_hero_healing',
    'players_5_hero_healing', 'players_6_hero_healing', 'players_7_hero_healing', 'players_8_hero_healing', 'players_9_hero_healing',

    # Gold and gold spent
    'players_0_gold', 'players_1_gold', 'players_2_gold', 'players_3_gold', 'players_4_gold',
    'players_5_gold', 'players_6_gold', 'players_7_gold', 'players_8_gold', 'players_9_gold',
    'players_0_gold_spent', 'players_1_gold_spent', 'players_2_gold_spent', 'players_3_gold_spent', 'players_4_gold_spent',
    'players_5_gold_spent', 'players_6_gold_spent', 'players_7_gold_spent', 'players_8_gold_spent', 'players_9_gold_spent',

    # Benchmarks: gold per minute, XP per minute, kills, last hits, hero damage, hero healing, tower damage
    'players_0_benchmarks_gold_per_min_raw', 'players_1_benchmarks_gold_per_min_raw', 'players_2_benchmarks_gold_per_min_raw',
    'players_3_benchmarks_gold_per_min_raw', 'players_4_benchmarks_gold_per_min_raw', 'players_5_benchmarks_gold_per_min_raw',
    'players_6_benchmarks_gold_per_min_raw', 'players_7_benchmarks_gold_per_min_raw', 'players_8_benchmarks_gold_per_min_raw',
    'players_9_benchmarks_gold_per_min_raw',
    'players_0_benchmarks_xp_per_min_raw', 'players_1_benchmarks_xp_per_min_raw', 'players_2_benchmarks_xp_per_min_raw',
    'players_3_benchmarks_xp_per_min_raw', 'players_4_benchmarks_xp_per_min_raw', 'players_5_benchmarks_xp_per_min_raw',
    'players_6_benchmarks_xp_per_min_raw', 'players_7_benchmarks_xp_per_min_raw', 'players_8_benchmarks_xp_per_min_raw',
    'players_9_benchmarks_xp_per_min_raw',
    'players_0_benchmarks_kills_per_min_raw', 'players_1_benchmarks_kills_per_min_raw', 'players_2_benchmarks_kills_per_min_raw',
    'players_3_benchmarks_kills_per_min_raw', 'players_4_benchmarks_kills_per_min_raw', 'players_5_benchmarks_kills_per_min_raw',
    'players_6_benchmarks_kills_per_min_raw', 'players_7_benchmarks_kills_per_min_raw', 'players_8_benchmarks_kills_per_min_raw',
    'players_9_benchmarks_kills_per_min_raw',
    'players_0_benchmarks_last_hits_per_min_raw', 'players_1_benchmarks_last_hits_per_min_raw', 'players_2_benchmarks_last_hits_per_min_raw',
    'players_3_benchmarks_last_hits_per_min_raw', 'players_4_benchmarks_last_hits_per_min_raw', 'players_5_benchmarks_last_hits_per_min_raw',
    'players_6_benchmarks_last_hits_per_min_raw', 'players_7_benchmarks_last_hits_per_min_raw', 'players_8_benchmarks_last_hits_per_min_raw',
    'players_9_benchmarks_last_hits_per_min_raw',
    'players_0_benchmarks_hero_damage_per_min_raw', 'players_1_benchmarks_hero_damage_per_min_raw', 'players_2_benchmarks_hero_damage_per_min_raw',
    'players_3_benchmarks_hero_damage_per_min_raw', 'players_4_benchmarks_hero_damage_per_min_raw', 'players_5_benchmarks_hero_damage_per_min_raw',
    'players_6_benchmarks_hero_damage_per_min_raw', 'players_7_benchmarks_hero_damage_per_min_raw', 'players_8_benchmarks_hero_damage_per_min_raw',
    'players_9_benchmarks_hero_damage_per_min_raw',
    'players_0_benchmarks_hero_healing_per_min_raw', 'players_1_benchmarks_hero_healing_per_min_raw', 'players_2_benchmarks_hero_healing_per_min_raw',
    'players_3_benchmarks_hero_healing_per_min_raw', 'players_4_benchmarks_hero_healing_per_min_raw', 'players_5_benchmarks_hero_healing_per_min_raw',
    'players_6_benchmarks_hero_healing_per_min_raw', 'players_7_benchmarks_hero_healing_per_min_raw', 'players_8_benchmarks_hero_healing_per_min_raw',
    'players_9_benchmarks_hero_healing_per_min_raw',
    'players_0_benchmarks_tower_damage_raw', 'players_1_benchmarks_tower_damage_raw', 'players_2_benchmarks_tower_damage_raw',
    'players_3_benchmarks_tower_damage_raw', 'players_4_benchmarks_tower_damage_raw', 'players_5_benchmarks_tower_damage_raw',
    'players_6_benchmarks_tower_damage_raw', 'players_7_benchmarks_tower_damage_raw', 'players_8_benchmarks_tower_damage_raw',
    'players_9_benchmarks_tower_damage_raw'
]

def data():
    matches = DotA_Data()

    #no_matches = 40560
    #temp_matches = matches.getData(no_matches)

    #full_matches = matches.flattenDataframe()

def Plot_Metric_vs_Features(num_features, accuracies_nn, accuracies_lstm, ylabel, title, fileName):
    plt.plot(num_features, accuracies_nn, label='Neural Network', color='teal')
    plt.plot(num_features, accuracies_lstm, label='LSTM', color='orange')

    plt.xlabel('Number of Features')
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()

    plt.savefig(fileName)

    plt.show()

def train_NN():

    # Load and prepare the dataset
    dota_data = DotA_Data()
    one_hot_encoded_dataset = dota_data.prepare_dataset(
        dataset_path=json_file_path,
        hero_id_columns=hero_id_columns,
        is_radiant_columns=is_radiant_columns,
        extra_columns=extra_columns,
        chunksize=chunksize
    )

    ################## NN Baseline Hyperparameters ################
    train_size = 0.6
    epochs = 150
    batch_size = 64
    learning_rate = 0.005

    ###############################################################
    print("Neural Network with all features")
    nn = DotA_NN()
    nn.train(one_hot_encoded_dataset, train_size, epochs, batch_size, learning_rate)

def train_BiLSTM():
    # Load and prepare the dataset
    dota_data = DotA_Data()
    one_hot_encoded_dataset = dota_data.prepare_dataset(
        dataset_path=json_file_path,
        hero_id_columns=hero_id_columns,
        is_radiant_columns=is_radiant_columns,
        extra_columns=extra_columns,
        chunksize=chunksize
    )

    ################## Stacked BiLSTM Baseline Hyperparameters ################
    train_size = 0.8
    epochs = 200
    batch_size = 64

    ######################################################
    print("Stacked BiLSTM with all features")
    lstm = DotA_Stacked_BiLSTM()
    lstm.train(one_hot_encoded_dataset, train_size, epochs, batch_size)


def postgreSqlExplorer():
    no_matches = 40560
    rate_limit = 1200
    backup_frequency = 100

    sql_matches = DotA_Postgre_Data()

    #match_ids_data = sql_matches.getMatchIDs(no_matches)

    match_data = sql_matches.getData(no_matches, rate_limit, backup_frequency)

def evaluateModels():
    num_features_list = []

    accuracies_nn = []
    accuracies_lstm = []

    loss_nn = []
    loss_lstm = []

    precision_nn = []
    precision_lstm = []

    roc_nn = []
    roc_lstm = []

    for i in range(10, len(extra_columns) + 1, 10):
        feature_subset = extra_columns[:i]
        num_features = len(feature_subset)
        print("Features: ", num_features, " out of ", len(extra_columns))
        num_features_list.append(num_features)

        chunksize = 100

        dota_data = DotA_Data()
        one_hot_encoded_dataset = dota_data.prepare_dataset(
            dataset_path=json_file_path,
            hero_id_columns=hero_id_columns,
            is_radiant_columns=is_radiant_columns,
            extra_columns=feature_subset,
            chunksize=chunksize
        )

        ####################### NN Evaluation ########################
        train_size = 0.6
        epochs = 150
        batch_size = 64
        learning_rate = 0.005

        nn = DotA_NN()

        test_accuracy, roc_auc, precision, test_loss = nn.train(one_hot_encoded_dataset, train_size, epochs, batch_size, learning_rate)
        accuracies_nn.append(test_accuracy)
        loss_nn.append(test_loss)
        precision_nn.append(precision)
        roc_nn.append(roc_auc)

        ####################### Stacked BiLSTM Evaluation ########################
        train_size = 0.8
        epochs = 200
        batch_size = 64

        lstm = DotA_Stacked_BiLSTM()

        test_accuracy, roc_auc, precision, test_loss = lstm.train(one_hot_encoded_dataset, train_size, epochs, batch_size)
        accuracies_lstm.append(test_accuracy)
        loss_lstm.append(test_loss)
        precision_lstm.append(precision)
        roc_lstm.append(roc_auc)

        np.save("num_features_list.npy", np.array(num_features_list))
        np.save("accuracies_nn.npy", np.array(accuracies_nn))
        np.save("accuracies_lstm.npy", np.array(accuracies_lstm))
        np.save("loss_nn.npy", np.array(loss_nn))
        np.save("loss_lstm.npy", np.array(loss_lstm))
        np.save("precision_nn.npy", np.array(precision_nn))
        np.save("precision_lstm.npy", np.array(precision_lstm))
        np.save("roc_nn.npy", np.array(roc_nn))
        np.save("roc_lstm.npy", np.array(roc_lstm))

    Plot_Metric_vs_Features(num_features_list, accuracies_nn, accuracies_lstm, ylabel='Accuracy', title='Comparison of Accuracies: NN vs LSTM', fileName='Comparison_of_Accuracies_NN_vs_LSTM.png')
    Plot_Metric_vs_Features(num_features_list, loss_nn, loss_lstm, ylabel='Loss', title='Comparison of Losses: NN vs LSTM', fileName='Comparison_of_Losses_NN_vs_LSTM.png')
    Plot_Metric_vs_Features(num_features_list, precision_nn, precision_lstm, ylabel='Precision', title='Comparison of Precisions: NN vs LSTM', fileName='Comparison_of_Precisions_NN_vs_LSTM.png')
    Plot_Metric_vs_Features(num_features_list, roc_nn, roc_lstm, ylabel='ROC AUC', title='Comparison of ROC AUC: NN vs LSTM', fileName='Comparison_of_ROC_AUC_NN_vs_LSTM.png')

    print("Model Evaluation Complete.")

def evaluateTrainedModels():
    num_features_list = np.load("num_features_list.npy")
    accuracies_nn = np.load("accuracies_nn.npy")
    accuracies_lstm = np.load("accuracies_lstm.npy")
    loss_nn = np.load("loss_nn.npy")
    loss_lstm = np.load("loss_lstm.npy")
    precision_nn = np.load("precision_nn.npy")
    precision_lstm = np.load("precision_lstm.npy")
    roc_nn = np.load("roc_nn.npy")
    roc_lstm = np.load("roc_lstm.npy")

    Plot_Metric_vs_Features(num_features_list, accuracies_nn, accuracies_lstm, ylabel='Accuracy', title='Comparison of Accuracies: NN vs LSTM', fileName='Comparison_of_Accuracies_NN_vs_LSTM.png')
    Plot_Metric_vs_Features(num_features_list, loss_nn, loss_lstm, ylabel='Loss', title='Comparison of Losses: NN vs LSTM', fileName='Comparison_of_Losses_NN_vs_LSTM.png')
    Plot_Metric_vs_Features(num_features_list, precision_nn, precision_lstm, ylabel='Precision', title='Comparison of Precisions: NN vs LSTM', fileName='Comparison_of_Precisions_NN_vs_LSTM.png')
    Plot_Metric_vs_Features(num_features_list, roc_nn, roc_lstm, ylabel='ROC AUC', title='Comparison of ROC AUC: NN vs LSTM', fileName='Comparison_of_ROC_AUC_NN_vs_LSTM.png')

def main():
    #postgreSqlExplorer()
    #data()
    #train_NN()
    #train_BiLSTM()
    evaluateModels()
    #evaluateTrainedModels()

if __name__ == "__main__":
    main()

Features:  10  out of  250
Processing data in chunks to build TensorFlow dataset
Number of devices: 1
Num GPUs Available:  1
First row saved to first_row.txt
Splitting data
First row saved to first_row_after_split.txt
Train dataset size: 24335
Val dataset size: 8111
Test dataset size: 8113
Begin model training
Model training complete
Model saved as 'dota_nn_model.keras'
Hyperparameter settings: 
 train_size:  0.6 
 epochs:  150 
 batch_size:  64 
 learning rate:  0.005

Test accuracy: 0.6438491940498352

Loss: 0.6142153143882751
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step  
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
2/2 ━━━━

/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Begin Training
Stacked BiLSTM Model training complete
Model saved as 'dota_lstm_model.keras'
Hyperparameter settings: 
 train_size:  0.8 
 epochs:  200 
 batch_size:  64

Test accuracy: 0.6289682388305664

Loss: 0.7695677876472473

Precision: 0.6301608

ROC AUC: 0.6726299
Features:  20  out of  250
Processing data in chunks to build TensorFlow dataset
Number of devices: 1
Num GPUs Available:  1
First row saved to first_row.txt
Splitting data
First row saved to first_row_after_split.txt
Train dataset size: 24335
Val dataset size: 8111
Test dataset size: 8113
Begin model training
Model training complete
Model saved as 'dota_nn_model.keras'
Hyperparameter settings: 
 train_size:  0.6 
 epochs:  150 
 batch_size:  64 
 learning rate:  0.005

Test accuracy: 0.5226934552192688

Loss: 0.6921361088752747
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step  
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2

/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Begin Training
Stacked BiLSTM Model training complete
Model saved as 'dota_lstm_model.keras'
Hyperparameter settings: 
 train_size:  0.8 
 epochs:  200 
 batch_size:  64

Test accuracy: 0.6966766119003296

Loss: 0.5858563184738159

Precision: 0.6562724

ROC AUC: 0.7593644
Features:  30  out of  250
Processing data in chunks to build TensorFlow dataset
Number of devices: 1
Num GPUs Available:  1
First row saved to first_row.txt
Splitting data
First row saved to first_row_after_split.txt
Train dataset size: 24335
Val dataset size: 8111
Test dataset size: 8113
Begin model training
Model training complete
Model saved as 'dota_nn_model.keras'
Hyperparameter settings: 
 train_size:  0.6 
 epochs:  150 
 batch_size:  64 
 learning rate:  0.005

Test accuracy: 0.5226934552192688

Loss: 0.6921361088752747
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step  
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2

...

Begin Training
Stacked BiLSTM Model training complete
Model saved as 'dota_lstm_model.keras'
Hyperparameter settings: 
 train_size:  0.8 
 epochs:  200 
 batch_size:  64

Test accuracy: 0.8216766119003296

Loss: 0.3963989019393921

Precision: 0.8986095

ROC AUC: 0.92098475
Features:  90  out of  250
Processing data in chunks to build TensorFlow dataset
Number of devices: 1
Num GPUs Available:  1
First row saved to first_row.txt
Splitting data
First row saved to first_row_after_split.txt
Train dataset size: 24335
Val dataset size: 8111
Test dataset size: 8113
Begin model training
Model training complete
Model saved as 'dota_nn_model.keras'
Hyperparameter settings: 
 train_size:  0.6 
 epochs:  150 
 batch_size:  64 
 learning rate:  0.005

Test accuracy: 0.5226934552192688

Loss: 0.6921361088752747
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step  
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Begin Training
Stacked BiLSTM Model training complete
Model saved as 'dota_lstm_model.keras'
Hyperparameter settings: 
 train_size:  0.8 
 epochs:  200 
 batch_size:  64

Test accuracy: 0.894345223903656

Loss: 0.26069605350494385

Precision: 0.9033349

ROC AUC: 0.9593682
Features:  100  out of  250
Processing data in chunks to build TensorFlow dataset
Number of devices: 1
Num GPUs Available:  1
First row saved to first_row.txt
Splitting data
First row saved to first_row_after_split.txt
Train dataset size: 24335
Val dataset size: 8111
Test dataset size: 8113
Begin model training
Model training complete
Model saved as 'dota_nn_model.keras'
Hyperparameter settings: 
 train_size:  0.6 
 epochs:  150 
 batch_size:  64 
 learning rate:  0.005

Test accuracy: 0.5226934552192688

Loss: 0.6921361088752747
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step  
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Begin Training
